# AgriEngineering — Revisión por pares | Validación con Rice Cammeo–Osmancik
**Manuscrito:** *Geometric Invariance and Representation Sufficiency in Morphometric Bean and Rice Classification*  
**Objetivo:** responder al comentario 3 del Revisor 1 con una validación en un conjunto externo de arroz (3.810 granos).  

**Precaución metodológica:** este notebook **reentrena** modelos para Cammeo/Osmancik; no transfiere los clasificadores de INIAP-11/12/20. Se conserva intacta la base y las particiones originales del artículo. Resultados exploratorios solicitados durante revisión, no confirmatorios preespecificados.

## 1. Librerías y fuente
La primera celda recibe el Excel aportado por los autores. Mantener el archivo original sin cambios; los análisis se hacen en memoria. Puede ejecutarse en Google Colab.

In [ ]:
import sys, subprocess
try:
    import pandas as pd
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pandas', 'openpyxl'])
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
from pathlib import Path
import hashlib
SEED = 20260803
from google.colab import drive
drive_root=Path('/content/drive/MyDrive')
if not drive_root.exists():
    drive.mount('/content/drive')
project_folder=drive_root/'Grain_Robustness_Q1'/'Revision_4664292_Ensayos_2026'
source_in_drive=project_folder/'Rice_Cammeo_Osmancik.xlsx'
filename=str(source_in_drive if source_in_drive.is_file() else Path('Rice_Cammeo_Osmancik.xlsx'))
if not Path(filename).is_file():
    from google.colab import files
    uploaded=files.upload()
    filename='Rice_Cammeo_Osmancik.xlsx'
    if filename not in uploaded:
        raise FileNotFoundError('Cargue Rice_Cammeo_Osmancik.xlsx')
RESULTS_DIR=project_folder/'resultados_colab'
RESULTS_DIR.mkdir(parents=True,exist_ok=True)
print('Dispositivo recomendado: CPU — SVM tabular, no GPU.')
print('Fuente:', filename, 'SHA-256:', hashlib.sha256(Path(filename).read_bytes()).hexdigest())

## 2. Auditoría de variables, clases y duplicados
La clasificación se realiza únicamente sobre características tabulares. Si aparecen duplicados exactos, el experimento se detiene para exigir una división por grupos y evitar fuga de información.

In [ ]:
raw = pd.read_excel(filename, sheet_name='Rice')
print('Columnas:', raw.columns.tolist())
assert raw.shape[0] == 3810
required = ['Area','Perimeter','Major_Axis_Length','Minor_Axis_Length','Eccentricity','Convex_Area','Extent','Class']
assert all(name in raw.columns for name in required)
raw = raw[required].copy()
assert raw.notna().all().all()
print('Registros por clase:')
print(raw['Class'].value_counts().sort_index())
print('Duplicados exactos (7 predictores):', raw.duplicated(subset=required[:7]).sum())
assert not raw.duplicated(subset=required[:7]).any(), 'Se requieren splits con grupos de duplicados.'
X = raw[required[:7]].to_numpy(float)
y = raw['Class'].to_numpy(str)

## 3. Congelar entrenamiento, calibración y prueba (60/20/20)
Se conserva la fracción de calibración sin utilizar: este análisis se centra en clasificación y representaciones y **no** pretende implementar APS ni calibración de probabilidad.

In [ ]:
indices = np.arange(len(y))
train_ids, rem_ids = train_test_split(indices, train_size=.60, stratify=y, random_state=SEED)
cal_ids, test_ids = train_test_split(rem_ids, test_size=.50, stratify=y[rem_ids], random_state=SEED)
print('Train/cal/test:',len(train_ids),len(cal_ids),len(test_ids))
assert (len(train_ids),len(cal_ids),len(test_ids)) == (2286,762,762)
assert len(set(train_ids)&set(test_ids))==0

## 4. Construcción de R1, R2, R3-NR y R4-NR
Definiciones comunes a la versión revisada. R2 mantiene `EXTENT`, que es **adimensional**, y excluye `ECCENTRICITY`, que también es adimensional y redundante con los ejes. R3-NR contiene cinco cantidades invariantes. R4-NR agrega `log(AREA)` para restituir una señal de escala.

In [ ]:
A,P,major,minor,ecc,conv,extent = (X[:,i] for i in range(7))
assert np.all((A>0)&(P>0)&(major>0)&(minor>0)&(conv>0))
AR = major/minor
solidity = A/conv
circularity = 4*np.pi*A/(P**2)
ellipse_fill = 4*A/(np.pi*major*minor)
invariant = np.column_stack([AR,extent,solidity,circularity,ellipse_fill])
features = {
    'R1':X,
    'R2':X[:,[0,1,2,3,5,6]],
    'R3-NR':invariant,
    'R4-NR':np.column_stack([invariant,np.log(A)])
}
for name, arr in features.items(): print(name,arr.shape)

## 5. Entrenamiento y selección exclusivamente con Train
Se aplica el mismo espacio de hiperparámetros RBF-SVM para las cuatro representaciones. `StandardScaler` se ajusta dentro de cada fold para evitar fuga. Para reflejar la convención del manuscrito, la clase final se obtiene por `argmax` de probabilidades SVC.

In [ ]:
cv = StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)
rows=[];pred={}
for name, feat in features.items():
    pipe = Pipeline([('scaler',StandardScaler()),('svc',SVC(kernel='rbf',probability=True,random_state=SEED))])
    grid = GridSearchCV(pipe, {'svc__C':[1,10], 'svc__gamma':['scale',.1]}, cv=cv, scoring='f1_macro',n_jobs=1)
    grid.fit(feat[train_ids],y[train_ids])
    yhat = grid.best_estimator_.classes_[np.argmax(grid.predict_proba(feat[test_ids]),axis=1)]
    pred[name]=yhat
    rows.append(dict(representation=name,n_features=feat.shape[1],cv_macro_f1=grid.best_score_,best_C=grid.best_params_['svc__C'],best_gamma=grid.best_params_['svc__gamma'],test_macro_f1=f1_score(y[test_ids],yhat,average='macro'),test_accuracy=accuracy_score(y[test_ids],yhat),test_balanced_accuracy=balanced_accuracy_score(y[test_ids],yhat)))
metrics=pd.DataFrame(rows)
display(metrics.round(5))

## 6. Inferencia pareada (2.000 remuestreos)
La unidad es un grano; en esta tabla no se encontraron duplicados exactos. Los intervalos son exploratorios. Usar ±0,01 como sensibilidad estadística, **sin** equiparar automáticamente el criterio a un costo agronómico.

In [ ]:
rng=np.random.default_rng(20260804)
ytest=y[test_ids]
class_ids={k:np.where(ytest==k)[0] for k in np.unique(ytest)}
comparisons=[]
for name in ['R2','R3-NR','R4-NR']:
    delta=f1_score(ytest,pred['R1'],average='macro')-f1_score(ytest,pred[name],average='macro')
    boot=[]
    for _ in range(2000):
        inds=np.concatenate([rng.choice(v,len(v),replace=True) for v in class_ids.values()])
        boot.append(f1_score(ytest[inds],pred['R1'][inds],average='macro')-f1_score(ytest[inds],pred[name][inds],average='macro'))
    ci95=np.percentile(boot,[2.5,97.5]);ci90=np.percentile(boot,[5,95])
    comparisons.append({'R1_minus':name,'delta':delta,'ci95_low':ci95[0],'ci95_high':ci95[1], 'ci90_low':ci90[0],'ci90_high':ci90[1],'within_001':bool(ci90[0]>-.01 and ci90[1]<.01)})
comp=pd.DataFrame(comparisons)
display(comp.round(5))

## 7. Matrices de confusión y exportación
Una matriz independiente para cada representación. Conservar `metrics` y `comp` en el suplemento y adjuntar el notebook a la respuesta editorial como análisis adicional, no reemplazo de los resultados de INIAP.

In [ ]:
labels=sorted(np.unique(y))
for name, yp in pred.items():
    cm=confusion_matrix(ytest, yp, labels=labels)
    print("\n", name, "filas verdaderas/columnas predichas:", labels, "\n", cm)
metrics.to_csv(RESULTS_DIR/'NB16_Cammeo_representation_test_results.csv',index=False)
comp.to_csv(RESULTS_DIR/'NB16_Cammeo_paired_bootstrap.csv',index=False)
print('Archivos CSV guardados en:',RESULTS_DIR)

## Lectura científica (resultados esperados en la ejecución auditada)
Con el archivo suministrado se obtuvo aproximadamente: R1 = **0,9356**; R2 = **0,9342**; R3-NR = **0,7972**; R4-NR = **0,9329** en test macro-F1. El efecto de eliminar escala es notable; agregar log-área recupera gran parte del desempeño, **pero el intervalo de R1−R4-NR todavía cruza el límite +0,01**. Las cifras impresas por la ejecución del usuario son la fuente numérica definitiva para su propia reproducción.

**Limitaciones:** este benchmark representa variedades turcas y otro esquema de captura. No evalúa directamente nuevos lotes ni nuevos dispositivos INIAP, y la selección entre representaciones no debe volver a hacerse mirando el conjunto de prueba.

## 8. Exportación automática del paquete para revisión — Cammeo–Osmancik

**Ejecuta esta última celda** después de completar las anteriores. Se guardará un único ZIP en Drive y Colab iniciará su descarga automática. Envía aquí **únicamente** `NB16_CAMMEO_OSMANCIK_RESULTADOS_4664292.zip` para evitar confundirlo con `NB16_RESULTADOS_REVISORES_4664292.zip`, que pertenece al ensayo estadístico anterior.

**Contiene:** dos CSV principales, F1 por clase, matrices de confusión, un resumen técnico, los metadatos del ensayo y comprobaciones SHA-256. No incluye fotografías originales ni modifica el Excel de entrada. **Entorno recomendado: CPU.**

In [ ]:
# ================================================================
# 8. CREAR Y DESCARGAR EL PAQUETE CAMMEO–OSMANCIK (solo este ensayo)
# ================================================================
from pathlib import Path
import sys, json, hashlib, zipfile, shutil
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
import sklearn

# Comprobación: las celdas del experimento deben haberse ejecutado primero.
requeridas = ['metrics', 'comp', 'pred', 'ytest', 'labels', 'filename', 'RESULTS_DIR', 'train_ids', 'cal_ids', 'test_ids']
faltan = [nombre for nombre in requeridas if nombre not in globals()]
if faltan:
    raise RuntimeError('Faltan resultados del ensayo: ' + ', '.join(faltan) + '. Ejecuta todas las celdas anteriores.')

NOMBRE_PAQUETE = 'NB16_CAMMEO_OSMANCIK_RESULTADOS_4664292'
carpeta_paquete = Path(RESULTS_DIR) / NOMBRE_PAQUETE
carpeta_paquete.mkdir(parents=True, exist_ok=True)

# Generamos el paquete a partir de las variables de ESTA ejecución,
# sin tomar otros ZIP o archivos guardados por notebooks anteriores.
metricas_archivo = carpeta_paquete / '01_Cammeo_metricas_representaciones.csv'
bootstrap_archivo = carpeta_paquete / '02_Cammeo_bootstrap_pareado.csv'
metrics.to_csv(metricas_archivo, index=False, encoding='utf-8-sig')
comp.to_csv(bootstrap_archivo, index=False, encoding='utf-8-sig')

filas_clase, filas_confusion = [], []
etiquetas = list(labels)
for representacion in ['R1', 'R2', 'R3-NR', 'R4-NR']:
    if representacion not in pred:
        raise RuntimeError(f'Falta la representación {representacion} en pred. No se creó el ZIP.')
    y_pred = np.asarray(pred[representacion])
    precision, recall, f1, soporte = precision_recall_fscore_support(
        ytest, y_pred, labels=etiquetas, zero_division=0
    )
    matriz = confusion_matrix(ytest, y_pred, labels=etiquetas)
    for i, clase in enumerate(etiquetas):
        filas_clase.append(dict(representation=representacion, clase=clase,
                                precision=float(precision[i]), recall=float(recall[i]),
                                f1=float(f1[i]), soporte=int(soporte[i])))
        for j, predicha in enumerate(etiquetas):
            filas_confusion.append(dict(representation=representacion,
                                        clase_real=clase, clase_predicha=predicha,
                                        n=int(matriz[i, j])))

pd.DataFrame(filas_clase).to_csv(
    carpeta_paquete / '03_Cammeo_F1_por_clase.csv', index=False, encoding='utf-8-sig'
)
pd.DataFrame(filas_confusion).to_csv(
    carpeta_paquete / '04_Cammeo_matrices_confusion.csv', index=False, encoding='utf-8-sig'
)

meta = {
    'manuscript_id': 'agriengineering-4664292',
    'analysis': 'External validation on Rice (Cammeo and Osmancik)',
    'dataset_sha256': hashlib.sha256(Path(filename).read_bytes()).hexdigest(),
    'dataset_filename': Path(filename).name,
    'seed': int(SEED),
    'n_train': len(train_ids), 'n_calibration': len(cal_ids), 'n_test': len(test_ids),
    'representations': ['R1', 'R2', 'R3-NR', 'R4-NR'],
    'prediction_rule': 'argmax over SVC predict_proba',
    'execution_type': 'CPU recommended; scikit-learn',
    'python': sys.version.split()[0],
    'scikit_learn': sklearn.__version__,
    'note': 'External dataset: models trained on Cammeo/Osmancik, not transferred from INIAP Rice.'
}
(carpeta_paquete / '05_Cammeo_metadatos.json').write_text(
    json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8'
)
(carpeta_paquete / '00_LEEME.txt').write_text(
    'Paquete de revisión: CAMMEO–OSMANCIK | agriengineering-4664292\n'
    'Experimento de validación externa independiente del NB16 estadístico Dry Bean/INIAP.\n'
    'Archivos: métricas R1/R2/R3-NR/R4-NR, bootstrap pareado, F1 por clase, '
    'matrices de confusión, metadatos y comprobación SHA-256.\n'
    'No contiene imágenes ni modifica las particiones congeladas de INIAP/Dry Bean.\n',
    encoding='utf-8'
)

# Huellas de los archivos incluidos, excluyendo el propio inventario.
contenido = sorted(p for p in carpeta_paquete.iterdir() if p.is_file() and p.name != '06_Cammeo_SHA256.csv')
pd.DataFrame([
    {'archivo': p.name, 'sha256': hashlib.sha256(p.read_bytes()).hexdigest(), 'tamano_bytes': p.stat().st_size}
    for p in contenido
]).to_csv(carpeta_paquete / '06_Cammeo_SHA256.csv', index=False, encoding='utf-8-sig')

# ZIP con nombre ÚNICO: no confundir con el NB16 de análisis interno.
ruta_zip = Path(RESULTS_DIR) / (NOMBRE_PAQUETE + '.zip')
archivos = sorted(p for p in carpeta_paquete.iterdir() if p.is_file())
with zipfile.ZipFile(ruta_zip, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as paquete:
    for archivo in archivos:
        paquete.write(archivo, arcname=NOMBRE_PAQUETE + '/' + archivo.name)
with zipfile.ZipFile(ruta_zip) as verificacion:
    assert verificacion.testzip() is None, 'El ZIP falló la verificación de integridad.'
    assert len(verificacion.namelist()) == len(archivos), 'El número de archivos del ZIP no coincide.'
print('✅ Paquete listo:', ruta_zip)
print('✅ Archivos incluidos:', len(archivos))
print('✅ SHA-256 del ZIP:', hashlib.sha256(ruta_zip.read_bytes()).hexdigest())
print('✅ Conservado en Google Drive:', ruta_zip)
print('📤 ENVÍAME ESTE ZIP (no el NB16 de la prueba interna):', ruta_zip.name)

# Descargar desde el navegador. Solo se activa al ejecutar en Google Colab.
try:
    from google.colab import files as colab_files
    colab_files.download(str(ruta_zip))
except ImportError:
    print('Fuera de Colab: abre la ruta mostrada y descarga el ZIP manualmente.')
